In [ ]:
import os
import random
import sys
import copy
import subprocess

os.environ['PYTHONUNBUFFERED'] = '1'
print('Script starting')

# =============================================================================
# MPI SETUP - Multi-node data parallelism
# =============================================================================

from mpi4py import MPI

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

print(f'MPI initialized: Rank {rank}/{size}')

import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.swa_utils import AveragedModel, update_bn
import pandas as pd
import numpy as np
import shap
import matplotlib
matplotlib.use('Agg')

state   = str(sys.argv[1])
parting = str(sys.argv[2])

print('Script imports done')

from datetime import datetime
from scipy.stats import spearmanr

# =============================================================================
# HYPERPARAMETERS
# =============================================================================

HIDDEN       = 256
N_BLOCKS     = 1
DROPOUT      = 0.1
MAX_LR       = 3e-4
WEIGHT_DECAY = 0
HUBER_BETA   = 0.2

# =============================================================================
# GPU ASSIGNMENT
# =============================================================================

def get_num_gpus():
    """
    Detect number of GPUs visible to this rank.

    torch.cuda.device_count() is checked first because it respects
    CUDA_VISIBLE_DEVICES, which SLURM sets when --gpus-per-task=1 is used.
    In that case every rank sees exactly cuda:0, so local_rank resolves to 0.

    nvidia-smi -L is only used as a fallback before CUDA has been fully
    initialised, since it ignores CUDA_VISIBLE_DEVICES and would return the
    full node count on a DGX H200, giving a wrong local_rank.
    """
    n = torch.cuda.device_count()
    if n > 0:
        print(f'[Rank {rank}] torch sees {n} GPU(s)')
        return n
    try:
        output = subprocess.check_output(['nvidia-smi', '-L'],
                                         stderr=subprocess.DEVNULL)
        n = len(output.decode().strip().split('\n'))
        print(f'[Rank {rank}] nvidia-smi sees {n} GPU(s)')
        return n
    except Exception:
        print(f'[Rank {rank}] GPU detection failed, defaulting to 1')
        return 1


_num_gpus  = get_num_gpus()
local_rank = rank % _num_gpus
torch.cuda.set_device(local_rank)
device     = torch.device(f'cuda:{local_rank}')

print(f'Rank {rank}: Using device {device} '
      f'(local_rank={local_rank}, gpus_visible={_num_gpus})')
print(f'Rank {rank}: GPU name: {torch.cuda.get_device_name(device)}')

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 1337 + rank
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
g = torch.Generator()
g.manual_seed(SEED)

# =============================================================================
# TRAINING CONFIG
# =============================================================================

NUM_EPOCHS      = 60
WARMUP_EPOCHS   = 12
BATCH_SIZE      = 4096
PATIENCE        = 7
VAL_FRAC        = 0.10
SWA_TAIL_EPOCHS = 20
SWA_LR          = MAX_LR * 0.10

# =============================================================================
# FEATURE GROUP INDICES
# var2 = ['Clay','Sand','Silt','Elevation','Aspect','Slope',   <- static  [0-5]
#          'LAI','MODIS','ALB','Temp','LandC',                 <- dynamic [6-10]
#          'Month','Year']                                      <- temporal[11-12]
# =============================================================================

STATIC_IDX   = [0, 1, 2, 3, 4, 5]   # Clay Sand Silt Elevation Aspect Slope
DYNAMIC_IDX  = [6, 7, 8, 9, 10]     # LAI MODIS ALB Temp LandC
TEMPORAL_IDX = [11, 12]             # Month Year

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def rank_corr_ave(data, verifier, subject):
    """Spearman correlation on spatially-averaged monthly anomalies."""
    data = data.dropna().copy()

    month_ave = (data[[verifier, subject, 'Year', 'Month', 'PageName']]
                 .groupby(['Year', 'Month', 'PageName']).mean()
                 .reset_index())

    hist_month_ave = (month_ave[[verifier, subject, 'Month', 'PageName']]
                      .groupby(['Month', 'PageName']).mean()
                      .reset_index()
                      .rename(columns={verifier: verifier + '_M',
                                       subject:  subject  + '_M'}))

    data_t = data.merge(hist_month_ave, on=['PageName', 'Month'])
    data_t[verifier + '_A'] = data_t[verifier] - data_t[verifier + '_M']
    data_t[subject  + '_A'] = data_t[subject]  - data_t[subject  + '_M']

    d = (data_t.groupby(['Month', 'Year'])
               .agg(verifier_A=(verifier + '_A', 'mean'),
                    subject_A =(subject  + '_A', 'mean'))
               .reset_index())

    spearman_corr, spearman_p = spearmanr(d['verifier_A'], d['subject_A'])
    return spearman_corr, spearman_p


def compute_shap_values(model, X_tr, X_ts, device):
    """SHAP GradientExplainer on a trained PyTorch model."""
    model.eval()
    background  = torch.tensor(X_tr[:500], dtype=torch.float32, device=device)
    test_tensor = torch.tensor(X_ts[:500], dtype=torch.float32, device=device)
    explainer   = shap.GradientExplainer(model, background)
    vals        = explainer.shap_values(test_tensor)
    if isinstance(vals, list):
        vals = vals[0]
    return np.asarray(vals)


# noFE variant: no engineered features. Raw integer Month and Year go straight
# into the model via the temporal gate; BatchNorm1d inside each group gate
# handles scale. This is the ablation counterpart to the FE versions.


def make_loaders(X_tr, y_tr):
    """Split arrays into train / val DataLoaders."""
    n     = len(X_tr)
    idx   = np.random.RandomState(SEED).permutation(n)
    n_val = max(1, int(n * VAL_FRAC))
    tr_idx, val_idx = idx[n_val:], idx[:n_val]

    def _loader(X, y, shuffle):
        return DataLoader(
            TensorDataset(torch.tensor(X, dtype=torch.float32),
                          torch.tensor(y, dtype=torch.float32)),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            generator=g if shuffle else None,
            num_workers=0,
            pin_memory=True,
            drop_last=shuffle,
        )

    return (_loader(X_tr[tr_idx],  y_tr[tr_idx],  shuffle=True),
            _loader(X_tr[val_idx], y_tr[val_idx],  shuffle=False))


# =============================================================================
# MODEL DEFINITION - Grouped-Gate GLU MLP
#
# Three independent gating streams (static / dynamic / temporal) each apply
# per-group BatchNorm + a sigmoid gate before concatenating into the shared
# projection layer. This prevents static and temporal features from crowding
# out dynamic features at the input, giving each group equal footing without
# hardcoding any importance penalty.
# =============================================================================

class GroupGate(nn.Module):
    """Soft per-feature attention gate within one feature group.

    Applies a learned sigmoid mask over the normalised group input so the
    model can suppress within-group features that add noise without needing
    the downstream GLU blocks to do all the filtering.
    """
    def __init__(self, n_features: int):
        super().__init__()
        self.norm = nn.BatchNorm1d(n_features)
        self.gate = nn.Sequential(
            nn.Linear(n_features, n_features),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        return x * self.gate(x)


class GLUBlock(nn.Module):
    """Gated Linear Unit block — learns to suppress irrelevant activations."""
    def __init__(self, dim: int, dropout: float = 0.15):
        super().__init__()
        self.fc   = nn.Linear(dim, dim * 2, bias=False)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h        = self.fc(self.norm(x))
        gate, value = h.chunk(2, dim=-1)
        return residual + self.drop(torch.sigmoid(gate) * torch.relu(value))


class BigRunMLP(nn.Module):
    def __init__(self,
                 static_idx:   list,
                 dynamic_idx:  list,
                 temporal_idx: list,
                 hidden:  int   = 256,
                 n_blocks: int  = 4,
                 dropout: float = 0.15):
        super().__init__()

        self.static_idx   = static_idx
        self.dynamic_idx  = dynamic_idx
        self.temporal_idx = temporal_idx

        n_s = len(static_idx)
        n_d = len(dynamic_idx)
        n_t = len(temporal_idx)
        in_dim = n_s + n_d + n_t   # same total as original; no added params here

        self.static_gate   = GroupGate(n_s)
        self.dynamic_gate  = GroupGate(n_d)
        self.temporal_gate = GroupGate(n_t)

        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            *[GLUBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 4),
            nn.ReLU(),
            nn.Linear(hidden // 4, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xs = self.static_gate(x[:, self.static_idx])
        xd = self.dynamic_gate(x[:, self.dynamic_idx])
        xt = self.temporal_gate(x[:, self.temporal_idx])
        return self.head(self.blocks(self.proj(torch.cat([xs, xd, xt], dim=1))))


# =============================================================================
# TRAINING FUNCTION
# =============================================================================

def ML_PyTorch(X_ts, X_tr, y_tr):
    """Full training run with grouped-gate BigRunMLP and SWA tail."""
    train_loader, val_loader = make_loaders(X_tr, y_tr)

    print(f'Rank {rank}: Training started at {datetime.now()}')
    print(f'Rank {rank}: hidden={HIDDEN}, n_blocks={N_BLOCKS}, '
          f'dropout={DROPOUT:.3f}, lr={MAX_LR:.2e}, '
          f'weight_decay={WEIGHT_DECAY:.2e}, huber_beta={HUBER_BETA:.3f}')
    print(f'Rank {rank}: train={int(len(X_tr) * (1 - VAL_FRAC))}, '
          f'val={int(len(X_tr) * VAL_FRAC)}')
    print(f'Rank {rank}: static={len(STATIC_IDX)}f  '
          f'dynamic={len(DYNAMIC_IDX)}f  '
          f'temporal={len(TEMPORAL_IDX)}f')

    model = BigRunMLP(
        static_idx=STATIC_IDX,
        dynamic_idx=DYNAMIC_IDX,
        temporal_idx=TEMPORAL_IDX,
        hidden=HIDDEN,
        n_blocks=N_BLOCKS,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=MAX_LR,
                                  weight_decay=WEIGHT_DECAY,
                                  betas=(0.9, 0.99), eps=1e-7)

    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=WARMUP_EPOCHS)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, NUM_EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

    criterion = nn.SmoothL1Loss(beta=HUBER_BETA)

    best_val          = float('inf')
    best_state        = None
    epochs_no_improve = 0

    for epoch in tqdm(range(NUM_EPOCHS), desc=f'Rank {rank} Training'):
        # ── train ─────────────────────────────────────────────────────────────
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                loss = criterion(model(Xb).squeeze(), yb.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1
        scheduler.step()

        # ── validate ──────────────────────────────────────────────────────────
        model.eval()
        val_loss, n_vb = 0.0, 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                    val_loss += criterion(model(Xb).squeeze(),
                                          yb.squeeze()).item()
                n_vb += 1
        val_loss /= max(1, n_vb)

        if val_loss < best_val - 1e-7:
            best_val          = val_loss
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
            print(f'Rank {rank}, Epoch {epoch:3d}, '
                  f'train={epoch_loss / max(1, n_batches):.6f}, '
                  f'val={val_loss:.6f}, '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}, '
                  f'patience={epochs_no_improve}/{PATIENCE}')

        if epochs_no_improve >= PATIENCE:
            print(f'Rank {rank}: Early stop at epoch {epoch} '
                  f'(best val={best_val:.6f})')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'Rank {rank}: Restored best weights (val={best_val:.6f})')

    # ── SWA tail ──────────────────────────────────────────────────────────────
    print(f'Rank {rank}: Starting SWA tail '
          f'({SWA_TAIL_EPOCHS} epochs at lr={SWA_LR:.2e})')
    swa_model = AveragedModel(model)
    for pg in optimizer.param_groups:
        pg['lr'] = SWA_LR

    for swa_epoch in range(SWA_TAIL_EPOCHS):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                loss = criterion(model(Xb).squeeze(), yb.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1
        swa_model.update_parameters(model)
        if swa_epoch % 3 == 0 or swa_epoch == SWA_TAIL_EPOCHS - 1:
            print(f'Rank {rank}: SWA epoch {swa_epoch:2d}, '
                  f'loss={epoch_loss / max(1, n_batches):.6f}')

    print(f'Rank {rank}: Updating BN stats for SWA model...')
    update_bn(train_loader, swa_model, device=device)

    # Sanity-check: did SWA help on val?
    swa_model.eval()
    swa_val, n_vb = 0.0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                swa_val += criterion(swa_model(Xb).squeeze(),
                                     yb.squeeze()).item()
            n_vb += 1
    swa_val /= max(1, n_vb)
    print(f'Rank {rank}: SWA val={swa_val:.6f} vs '
          f'best pre-SWA val={best_val:.6f}')

    model.load_state_dict(swa_model.module.state_dict())
    print(f'Rank {rank}: SWA weights loaded. '
          f'Training complete at {datetime.now()}')

    # ── Batched inference ─────────────────────────────────────────────────────
    model.eval()
    predictions = []
    with torch.no_grad():
        torch.cuda.empty_cache()
        for i in range(0, len(X_ts), BATCH_SIZE):
            Xb    = torch.tensor(X_ts[i:i + BATCH_SIZE],
                                 dtype=torch.float32).to(device)
            pred_b = model(Xb).cpu().numpy()
            predictions.append(pred_b)
            del Xb
            torch.cuda.empty_cache()

    pred = np.concatenate(predictions, axis=0).ravel()
    return pred, model


# =============================================================================
# MAIN EXECUTION
# =============================================================================

home       = '/scratch/project/prj-16-pi-kenneth-tobin'
resolution = (f'/scratch/user/aaron.sanchez_tamiu.edu/'
              f'GLU_{state}_500v10noFELC-20140730-{rank}')
train_file = f'{home}/{state}_trainV6-LC.csv'
test_file  = f'{home}/{state}_testV6-LC.csv'

if rank == 0:
    print(f'\nMPI Multi-Node GLU Configuration:')
    print(f'  Total ranks  : {size}')
    print(f'  GPUs visible : {_num_gpus}')
    print(f'  State        : {state}')
    print(f'  hidden       : {HIDDEN}')
    print(f'  n_blocks     : {N_BLOCKS}')
    print(f'  dropout      : {DROPOUT:.3f}')
    print(f'  lr           : {MAX_LR:.2e}')
    print(f'  weight_decay : {WEIGHT_DECAY:.2e}')
    print(f'  huber_beta   : {HUBER_BETA:.3f}')
    print(f'  Batch size   : {BATCH_SIZE}')
    print(f'  Loading from : {train_file}')

# ── Column sets ───────────────────────────────────────────────────────────────

all_vars = ['PageName', 'Clay', 'Sand', 'Silt', 'Elevation', 'Slope',
            'Aspect', 'MODIS', 'SMERGE', 'Date', 'LAI', 'ALB', 'Temp',
            'AHRR', 'Month', 'Year', 'LandC']

# noFE: raw integer Month and Year feed the temporal gate directly.
# Group order must match STATIC_IDX / DYNAMIC_IDX / TEMPORAL_IDX above.
# var2 = ['Clay','Sand','Silt','Elevation','Aspect','Slope',   <- [0-5]  static
#          'LAI','MODIS','ALB','Temp','LandC',                 <- [6-10] dynamic
#          'Month','Year']                                      <- [11-12] temporal
var2 = ['Clay', 'Sand', 'Silt', 'Elevation', 'Aspect', 'Slope',
        'LAI', 'MODIS', 'ALB', 'Temp', 'LandC',
        'Month', 'Year']

# ── Load + clean ──────────────────────────────────────────────────────────────

def load_and_clean(path):
    df = pd.read_csv(path, usecols=all_vars, engine='pyarrow').dropna()
    df['Soil'] = df['Sand'] + df['Silt'] + df['Clay']
    df = df[(df['Soil'] >= 0.9999) & (df['Soil'] < 1.0001)].reset_index()
    for col in ['index', 'level_0']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df.dropna()


train_df = load_and_clean(train_file)
test_df  = load_and_clean(test_file)

print(f'Rank {rank}: Data loaded — Train={len(train_df)}, Test={len(test_df)}')

# ── Strided per-rank split ─────────────────────────────────────────────────────

data_chunk_train = train_df.iloc[rank::size].reset_index(drop=True)
data_chunk_test  = test_df.iloc[rank::size].reset_index(drop=True)

print(f'Rank {rank}: Chunk {rank}/{size} — '
      f'Train={len(data_chunk_train)}, Test={len(data_chunk_test)}')

X_train = data_chunk_train[var2].astype(np.float32).values
y_train = data_chunk_train[['SMERGE']].astype(np.float32).values
X_test  = data_chunk_test[var2].astype(np.float32).values

print(f'Rank {rank}: X_train shape={X_train.shape}')

# =============================================================================
# TRAIN + PREDICT
# =============================================================================

ml_out, trained_model = ML_PyTorch(X_test, X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────

data_chunk_test['ML_'] = ml_out
corr, p_value = rank_corr_ave(data_chunk_test, 'AHRR', 'ML_')
print(f'Rank {rank}: Spearman corr={corr:.4f}, p={p_value:.4e}')

# ── Save predictions ──────────────────────────────────────────────────────────

output_path = (f'{resolution}_'
               f'{int(rank)}.csv')
try:
    import dask.dataframe as dd
    ddf = dd.from_pandas(data_chunk_test, npartitions=12)
    ddf.to_csv(output_path, single_file=True, index=False)
    print(f'Rank {rank}: Results saved via Dask -> {output_path}')
except Exception as e:
    print(f'Rank {rank}: Dask failed ({e}), falling back to pandas')
    data_chunk_test.to_csv(output_path, index=False)

# =============================================================================
# SHAP FEATURE IMPORTANCE
# =============================================================================

print(f'Rank {rank}: Computing SHAP values...')
raw_shap = compute_shap_values(trained_model, X_train, X_test, device)
raw_shap = np.squeeze(raw_shap, axis=-1)

feat_importance = np.mean(np.abs(raw_shap), axis=0)
shap_df = (pd.DataFrame({'Feature': var2, 'Mean|SHAP|': feat_importance})
             .sort_values('Mean|SHAP|', ascending=False))

shap_path = (f'{resolution}_'
             f'{int(rank)}'
             f'_shap_importance.csv')
shap_df.to_csv(shap_path, index=False)
print(f'Rank {rank}: SHAP table -> {shap_path}')
print(shap_df.to_string(index=False))

if rank == 0:
    print('\n' + '=' * 60)
    print('All ranks completed successfully!')
    print('=' * 60)
